# FloodOps SG — RL Flood Prediction

Trains a DQN agent in a simulated flood environment, exports to `.onnx` for browser inference.

**Pipeline:**
1. Install deps
2. Define the Gym environment (matches live sensor features in the app)
3. Train DQN → `flood_policy.pth`
4. Export → `flood_policy.onnx`
5. Copy `flood_policy.onnx` to `public/` in the repo

## 1 — Install dependencies

In [1]:
%pip install gymnasium torch numpy --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 — Gym Environment

**State (6 features):**
| Feature | Source in app | Normalisation |
|---------|--------------|---------------|
| `rainfallMm` | data.gov.sg `/rainfall` | ÷ 150 |
| `waterLevelPercent` | `zone.waterLevelPercent` | ÷ 100 |
| `trend_rising` | derived from rainfall delta | one-hot |
| `trend_stable` | derived from rainfall delta | one-hot |
| `trend_falling` | derived from rainfall delta | one-hot |
| `official` | signal type === 'official' | 0 / 1 |

**Actions:** 0 = no alert · 1 = watch · 2 = warning · 3 = flash flood alert

**Reward:** correct early escalation = +10, false alarm = −8, missed flood = −20

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class FloodEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=np.zeros(6, dtype=np.float32),
            high=np.ones(6, dtype=np.float32),
        )
        self.action_space = spaces.Discrete(4)
        self.max_steps = 60

    def _obs(self):
        trend_vec = [0.0, 0.0, 0.0]
        trend_vec[["rising", "stable", "falling"].index(self._trend)] = 1.0
        return np.array([
            self._rain  / 150.0,
            self._water / 100.0,
            *trend_vec,
            float(self._official),
        ], dtype=np.float32)

    def _risk(self):
        # mirrors zoneRiskScore() in src/lib/brief/score.ts
        return (0.4 * min(self._rain  / 100.0, 1.0)
              + 0.3 * min(self._water / 100.0, 1.0)
              + 0.1 * float(self._official))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._rain     = self.np_random.uniform(0, 80)
        self._water    = self.np_random.uniform(10, 60)
        self._trend    = self.np_random.choice(["rising", "stable", "falling"])
        self._official = self.np_random.random() < 0.1
        self._step     = 0
        self._prev_rain = self._rain
        return self._obs(), {}

    def step(self, action):
        self._step += 1

        # Rainfall dynamics
        delta = (self.np_random.uniform(2,  10) if self._trend == "rising"  else
                 self.np_random.uniform(-8,  0) if self._trend == "falling" else
                 self.np_random.uniform(-3,  3))
        self._rain  = float(np.clip(self._rain + delta, 0, 150))
        self._water = float(np.clip(
            self._water + (self.np_random.uniform(0, 3) if self._rain > 40
                           else self.np_random.uniform(-1, 1)), 0, 100))

        diff = self._rain - self._prev_rain
        self._trend = "rising" if diff > 3 else "falling" if diff < -3 else "stable"
        self._prev_rain = self._rain
        self._official = self._official or (
            self._rain > 80 and self.np_random.random() < 0.2)

        risk  = self._risk()
        flood = risk > 0.65 and self.np_random.random() < risk
        done  = flood or self._step >= self.max_steps

        if flood:
            reward = {3: 10.0, 2: 5.0, 1: 2.0, 0: -20.0}[action]
        else:
            if   action == 3 and risk < 0.3: reward = -8.0
            elif action == 2 and risk < 0.2: reward = -4.0
            elif action == 0 and risk < 0.3: reward =  0.5
            else:                             reward =  0.0

        return self._obs(), reward, done, False, {"flood": flood}

# quick sanity check
env = FloodEnv()
obs, _ = env.reset(seed=42)
print("obs shape:", obs.shape, "| sample obs:", obs.round(3))
obs2, r, done, _, info = env.step(0)
print("step reward:", r, "| done:", done, "| flood:", info["flood"])

## 3 — DQN Model

In [ ]:
import torch
import torch.nn as nn

class DQN(nn.Module):
    """6 inputs → 64 → 64 → 4 Q-values (one per alert level)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 4),
        )
    def forward(self, x):
        return self.net(x)

print(DQN())

## 4 — Training Loop

DQN with experience replay and a target network. Adjust `EPISODES` as needed — 5 000 converges in ~1 min on CPU.

In [ ]:
import random
import torch.optim as optim
from collections import deque

EPISODES   = 5_000
BATCH      = 64
LR         = 1e-3
GAMMA      = 0.95
EPS_START  = 1.0
EPS_MIN    = 0.01
EPS_DECAY  = 0.995
TARGET_UPD = 500   # episodes between target-network syncs
REPLAY_CAP = 10_000

env    = FloodEnv()
policy = DQN()
target = DQN()
target.load_state_dict(policy.state_dict())
target.eval()

optimizer = optim.Adam(policy.parameters(), lr=LR)
replay    = deque(maxlen=REPLAY_CAP)
epsilon   = EPS_START
rewards_window = deque(maxlen=200)

for ep in range(1, EPISODES + 1):
    obs, _ = env.reset()
    total  = 0.0

    while True:
        # epsilon-greedy action selection
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()

        next_obs, reward, done, _, _ = env.step(action)
        replay.append((obs, action, reward, next_obs, done))
        obs    = next_obs
        total += reward

        # train when we have enough samples
        if len(replay) >= BATCH:
            s, a, r, ns, d = zip(*random.sample(replay, BATCH))
            s  = torch.tensor(np.array(s),  dtype=torch.float32)
            a  = torch.tensor(a,             dtype=torch.long)
            r  = torch.tensor(r,             dtype=torch.float32)
            ns = torch.tensor(np.array(ns),  dtype=torch.float32)
            d  = torch.tensor(d,             dtype=torch.float32)

            q_pred   = policy(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                q_target = r + GAMMA * target(ns).max(1).values * (1 - d)

            loss = nn.functional.mse_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
    rewards_window.append(total)

    if ep % TARGET_UPD == 0:
        target.load_state_dict(policy.state_dict())

    if ep % 500 == 0:
        avg = sum(rewards_window) / len(rewards_window)
        print(f"Episode {ep:>5}/{EPISODES}  avg_reward={avg:+.2f}  eps={epsilon:.3f}")

print("\nTraining complete.")

## 5 — Save PyTorch Checkpoint

In [ ]:
torch.save(policy.state_dict(), "flood_policy.pth")
print("Saved flood_policy.pth")

## 6 — Export to ONNX

The `.onnx` file is what gets loaded in the browser via `onnxruntime-web`.

In [ ]:
policy.eval()
dummy = torch.zeros(1, 6)

torch.onnx.export(
    policy,
    dummy,
    "flood_policy.onnx",
    input_names=["state"],
    output_names=["q_values"],
    dynamic_axes={"state": {0: "batch"}, "q_values": {0: "batch"}},
    opset_version=17,
)
print("Exported flood_policy.onnx")

## 7 — Verify the ONNX model

In [ ]:
import onnxruntime as ort   # pip install onnxruntime if needed
import numpy as np

sess  = ort.InferenceSession("flood_policy.onnx")
dummy = np.array([[0.8, 0.7, 1.0, 0.0, 0.0, 1.0]], dtype=np.float32)  # heavy rain, rising, official
q     = sess.run(["q_values"], {"state": dummy})[0][0]

LABELS = ["no_alert", "watch", "warning", "flash_flood"]
best   = int(np.argmax(q))
print("Q-values:", dict(zip(LABELS, q.round(3))))
print("Predicted action:", LABELS[best])

## 8 — Next steps

1. Copy `flood_policy.onnx` into the repo's `public/` folder:
   ```
   cp flood_policy.onnx ../../public/
   ```

2. Install the npm package:
   ```
   npm install onnxruntime-web
   ```

3. Create `src/lib/flood/infer.ts`:

```typescript
import * as ort from 'onnxruntime-web';

export type Trend = 'rising' | 'stable' | 'falling';

export interface FloodInput {
  rainfallMm: number;
  waterLevelPercent: number;
  trend: Trend;
  official: boolean;
}

export interface AlertPrediction {
  action: 0 | 1 | 2 | 3;
  label: string;
  confidence: number;
  qValues: number[];
}

const LABELS = ['No Alert', 'Watch', 'Warning', 'Flash Flood Alert'] as const;
let session: ort.InferenceSession | null = null;

export async function loadFloodModel(): Promise<void> {
  session = await ort.InferenceSession.create('/flood_policy.onnx');
}

export async function predictFloodAlert(input: FloodInput): Promise<AlertPrediction> {
  if (!session) await loadFloodModel();

  const trendVec: [number, number, number] =
    input.trend === 'rising'  ? [1, 0, 0] :
    input.trend === 'stable'  ? [0, 1, 0] :
                                [0, 0, 1];

  const state = new Float32Array([
    input.rainfallMm / 150,
    input.waterLevelPercent / 100,
    ...trendVec,
    input.official ? 1 : 0,
  ]);

  const tensor  = new ort.Tensor('float32', state, [1, 6]);
  const results = await session!.run({ state: tensor });
  const qValues = Array.from(results['q_values'].data as Float32Array);

  const action = qValues.indexOf(Math.max(...qValues)) as 0 | 1 | 2 | 3;
  const max    = Math.max(...qValues);
  const min    = Math.min(...qValues);
  const conf   = max === min ? 0.5 : (qValues[action] - min) / (max - min);

  return {
    action,
    label: LABELS[action],
    confidence: Math.min(0.95, Math.max(0.5, conf)),
    qValues,
  };
}
```